Catastrophic Forgetting/Interference

Confirming and Visualizing the interference

In [28]:
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

Setting up Hyperparameters

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate = 1e-3
batch_size = 64
epochs_per_task = 3
tasks = [(0, 1), (2, 3), (4, 5), (6, 7), (8, 9)]

In [30]:
print(f"Using device: {device}")

Using device: cpu


Dataset Loading and splitting

In [31]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [32]:
train_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=True, download=True, transform=mnist_transform)
test_dataset = datasets.MNIST(root='/home/bukunmi/ml-journey/datasets/data', train=False, download=True, transform=mnist_transform)

In [33]:
# Filter by Digits

def filter_by_digits(dataset, digits):
    indices = [i for i, (_, label) in enumerate(dataset) if label in digits]
    return Subset(dataset, indices)

In [34]:
# Get split MNIST datasets for each task
def get_split_mnist_datasets(tasks_digits):
    train_subset = filter_by_digits(train_dataset, tasks_digits)
    test_subset = filter_by_digits(test_dataset, tasks_digits)
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

Instantiate Model, optimizer, criterion and task loaders

In [35]:
task_loaders = [get_split_mnist_datasets(task) for task in tasks]

#### PROGRESSIVE LAYER

In [36]:
class P_Layer(nn.Module):
    def __init__(self, fan_in, fan_out, active_idx):
        super().__init__()
        self.active_idx = active_idx
        self.W = nn.Linear(fan_in, fan_out)
        self.V = nn.ModuleList([nn.Linear(fan_in, fan_out) for _ in range(active_idx)])
    # prev_layers_outputs
    def forward(self, prev_layer_outputs):
        out = self.W(prev_layer_outputs[self.active_idx])
        for j in range(self.active_idx):
            out += self.V[j](prev_layer_outputs[j])
        return F.relu(out)

Progressive Neural Net

In [37]:
class P_Net(nn.Module):
    def __init__(self, layer_sizes=[784, 128, 64, 2]):
        super().__init__()
        self.layer_sizes = layer_sizes
        self.columns = nn.ModuleList([])

    def add_columns(self):
        new_idx = len(self.columns)
        for param in self.parameters():
            param.requires_grad = False
        
        column_layers = nn.ModuleList()
        for i in range(len(self.layer_sizes) - 1):
            if i < len(self.layer_sizes) - 2:
                column_layers.append(P_Layer(self.layer_sizes[i], self.layer_sizes[i+1], new_idx))
            else:
                column_layers.append(nn.Linear(self.layer_sizes[i], self.layer_sizes[i+1]))
        self.columns.append(column_layers)
    
    def forward(self, x, task_id):
        layer_outputs = [[x] for _ in range(task_id + 1)]

        for layer_idx in range(len(self.layer_sizes) - 2):
            previous_layer_outputs = [layer_outputs[j][-1] for j in range(task_id + 1)]
            for col_idx in range(task_id + 1):
                prev_outputs = previous_layer_outputs[:col_idx + 1]
                out = self.columns[col_idx][layer_idx](prev_outputs)
                layer_outputs[col_idx].append(out)
        
        final_head_idx = len(self.layer_sizes) - 2
        logits = self.columns[task_id][final_head_idx](layer_outputs[task_id][-1])
        return logits

Training and Evaluation Loop

In [38]:
def remap_labels(labels, task_digits):
    return (labels == task_digits[1]).long()

In [39]:
def prepare_batch(images, labels, task_digits):
    images = images.to(device).view(images.size(0), -1)
    labels = remap_labels(labels.to(device), task_digits)
    return images, labels

In [40]:
def evaluate(model, eval_loader, task_idx, task_digits):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in eval_loader:
            images, labels = prepare_batch(images, labels, task_digits)
            logits = model(images, task_idx)
            predicted = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (labels == predicted).sum().item()
    return correct / total

In [41]:
def train():
    model = P_Net().to(device)
    print(f"Starting training on {device} with {len(tasks)} tasks")
    
    for task_idx, ((train_loader, eval_loader), task_digits) in enumerate(zip(task_loaders, tasks)):
        model.add_columns()
        model.to(device)
        active_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.Adam(active_params, lr=learning_rate)

        print(f"\nTask {task_idx + 1}/{len(tasks)} digits={task_digits} columns={len(model.columns)} trainable_params={sum(p.numel() for p in active_params):,}")

        for epoch in range(epochs_per_task):
            model.train()
            running_loss, total = 0.0, 0
            for batch_idx, (images, labels) in enumerate(train_loader):
                images, labels = prepare_batch(images, labels, task_digits)
                if epoch == 0 and batch_idx == 0:
                    print(f"First batch images={tuple(images.shape)} labels={sorted(labels.unique().tolist())}")
                optimizer.zero_grad()
                output = model(images, task_idx)
                loss = F.cross_entropy(output, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * labels.size(0)
                total += labels.size(0)
            accuracy = evaluate(model, eval_loader, task_idx, task_digits)
            print(f"  epoch {epoch + 1}/{epochs_per_task} loss={running_loss / total:.4f} eval_acc={accuracy:.2%}")
    return model

In [42]:
train()

Starting training on cpu with 5 tasks

Task 1/5 digits=(0, 1) columns=1 trainable_params=108,866
First batch images=(64, 784) labels=[0, 1]


  epoch 1/3 loss=0.0211 eval_acc=99.86%
  epoch 2/3 loss=0.0028 eval_acc=99.91%
  epoch 3/3 loss=0.0033 eval_acc=99.95%

Task 2/5 digits=(2, 3) columns=2 trainable_params=217,602
First batch images=(64, 784) labels=[0, 1]
  epoch 1/3 loss=0.0835 eval_acc=98.92%
  epoch 2/3 loss=0.0315 eval_acc=99.56%
  epoch 3/3 loss=0.0184 eval_acc=99.41%

Task 3/5 digits=(4, 5) columns=3 trainable_params=326,338
First batch images=(64, 784) labels=[0, 1]
  epoch 1/3 loss=0.0431 eval_acc=99.89%
  epoch 2/3 loss=0.0093 eval_acc=99.95%
  epoch 3/3 loss=0.0027 eval_acc=99.84%

Task 4/5 digits=(6, 7) columns=4 trainable_params=435,074
First batch images=(64, 784) labels=[0, 1]
  epoch 1/3 loss=0.0160 eval_acc=99.85%
  epoch 2/3 loss=0.0038 eval_acc=99.95%
  epoch 3/3 loss=0.0014 eval_acc=99.90%

Task 5/5 digits=(8, 9) columns=5 trainable_params=543,810
First batch images=(64, 784) labels=[0, 1]
  epoch 1/3 loss=0.0638 eval_acc=98.29%
  epoch 2/3 loss=0.0291 eval_acc=98.89%
  epoch 3/3 loss=0.0158 eval_acc

P_Net(
  (columns): ModuleList(
    (0): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList()
      )
      (1): P_Layer(
        (W): Linear(in_features=128, out_features=64, bias=True)
        (V): ModuleList()
      )
      (2): Linear(in_features=64, out_features=2, bias=True)
    )
    (1): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList(
          (0): Linear(in_features=784, out_features=128, bias=True)
        )
      )
      (1): P_Layer(
        (W): Linear(in_features=128, out_features=64, bias=True)
        (V): ModuleList(
          (0): Linear(in_features=128, out_features=64, bias=True)
        )
      )
      (2): Linear(in_features=64, out_features=2, bias=True)
    )
    (2): ModuleList(
      (0): P_Layer(
        (W): Linear(in_features=784, out_features=128, bias=True)
        (V): ModuleList(
          (0-1): 2 x Linear(i